In [4]:
numbers = [10, 20, 30, 40, 50]
print(numbers[2])
print(len(numbers))
print(numbers == [])

30
5
False


In [ ]:
import torch
import torch.nn as nn
class TriangleMultiplicationOutgoing(nn.Module):
    def __init__(self,c_z=128,c_hidden=128):
        super().__init__()
        self.layer_norm_in = nn.LayerNorm(c_z)
        self.linear_a = nn.Linear(c_z,c_hidden)
        self.linear_a_gate = nn.Linear(c_z,c_hidden)
        self.linear_b = nn.Linear(c_z,c_hidden)
        self.linear_b_gate = nn.Linear(c_z,c_hidden)
        self.linear_g = nn.Linear(c_z,c_z)
        self.layer_norm_out = nn.LayerNorm(c_hidden)
        self.linear_out = nn.Linear(c_hidden,c_z)
    def forward(self,z):
        z_norm = self.layer_norm_in(z)
        a = self.linear_a(z_norm)
        a = a * torch.sigmoid(self.linear_a_gate(z_norm))
        b = self.linear_b(z_norm)
        b = b * torch.sigmoid(self.linear_b_gate(z_norm))
        x = torch.einsum('ikc,jkc->ijc',a,b)
        x = self.layer_norm_out(x)
        x = self.linear_out(x)
        gate = torch.sigmoid(self.linear_g(z_norm))
        return x * gate

In [ ]:
import torch
import torch.nn as nn
class TriangleMultiplicationIncoming(nn.Module):
    def __init__(self,c_z=128,c=128):
        super().__init__()
        self.layer_norm_in = nn.LayerNorm(c_z)
        self.linear_a_gate = nn.LayerNorm(c_z)
        self.linear_a_value = nn.Linear(c_z,c)
        self.linear_b_gate = nn.LayerNorm(c_z,c)
        self.linear_b_value = nn.Linear(c_z,c)
        self.linear_gate = nn.Linear(c_z,c_z)
        self.layer_norm_out = nn.LayerNorm(c)
        self.linear_out = nn.Linear(c,c_z)
    def forward(self,z):
        z = self.layer_norm(z)
        a = (torch.sigmoid(self.linear_a_gate(z))*self.linear_a_value(z))
        b = (torch.sigmoid(self.linear_b_gate(z))*self.linear_b_value(z))
        g = torch.sigmoid(self.linear(z))
        u = torch.einsum('bkic,bkjc->bijc',a,b)
        u = self.layer_norm_out(u)
        u = self.linear_out(u)
        u = g * u
        return u

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
class TriangleAttentionStartingNode(nn.Module):
    def __init__(self,c_z=128,num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = c_z // num_heads
        self.norm = nn.LayerNorm(c_z)
        self.q = nn.Linear(c_z,c_z,bias=False)
        self.k = nn.Linear(c_z,c_z,bias=False)
        self.v = nn.Linear(c_z,c_z,bias=False)
        self.bias = nn.Linear(c_z,num_heads,bias=False)
        self.gate = nn.Linear(c_z,num_heads,bias=False)
        self.output = nn.Linear(c_z,c_z)
    def forward(self,z,mask=None):
        N,_,C = z.shape
        H = self.num_heads
        D = self.head_dim
        z_norm = self.norm(z)
        q = self.q(z_norm).view(N,N,H,D)
        k = self.k(z_norm).view(N,N,H,D)
        v = self.v(z_norm).view(N,N,H,D)
        b = self.bias(z_norm)
        logits = torch.einsum("ijhd,ikhd->ijkh",q,k) / math.sqrt(D)
        logits = logits + b[None,:,:,:]
        if mask is not None:
            logits = logits.masked_fill(~mask[:,None,None,:].bool(),-1e9)
        attn = F.softmax(logits,dim=2)
        out = torch.einsum("ijkh,ikhd->ijhd",attn,v)
        gate = torch.sigmoid(self.gate(z_norm)).view(N,N,H,D)
        out = out * gate
        out = out.reshape(N,N,C)
        return self.output(out)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
class TriangleAttentionEndingNode(nn.Module):
    def __init__(self,c_z=128,num_heads=4,dropout=0.0):
        super().__init__()
        self.c_z = c_z
        self.num_heads = num_heads
        self.head_dim = c_z // num_heads
        assert c_z % num_heads == 0
        self.layer_norm = nn.LayerNorm(c_z)
        self.linear_q = nn.Linear(c_z,c_z,bias=False)
        self.linear_k = nn.Linear(c_z,c_z,bias=False)
        self.linear_v = nn.Linear(c_z,c_z,bias=False)
        self.linear_b = nn.Linear(c_z,num_heads,bias=False)
        self.linear_g = nn.Linear(c_z,c_z)
        self.linear_o = nn.Linear(c_z,c_z)
        self.dropout = dropout
    def forward(self,z,pair_mask=None):
        B,L,_,C = z.shape
        H = self.num_heads
        D = self.head_dim
        z_norm = self.layer_norm(z)
        q = self.linear_q(z_norm)
        k = self.linear_k(z_norm)
        v = self.linear_v(z_norm)
        q = q.reshape(B,L,L,H,D)
        k = k.reshape(B,L,L,H,D)
        v = v.reshape(B,L,L,H,D)
        q = q.permute(0,3,2,1,4)
        k = k.permute(0,3,2,1,4)
        v = v.permute(0,3,2,1,3)
        logits = torch.einsum("bhjid,bhjkd->bhjik",q,k)
        logits = logits / math.sqrt(D)
        bias = bias.permute(0,3,2,1)
        logits = logits + bias[:,:,None,:,:]
        if pair_mask is not None:
            key_mask = pair_mask.transpose(1,2)
            key_mask = key_mask[:,None,:,None,:]
            logits = logits.masked_fill(key_mask == 0,-1e9)
            attn = F.softmax(logits,dim=-1)
            out = torch.einsum("bhjik,bhjkd->bhjid",attn,v)
            out = out.permute(0,3,2,1,4)
            out = out.reshape(B,L,L,C)
            gate = torch.sigmoid(self.linear_g(z_norm))
            out = out * gate
            out = self.linear_o(out)
            return out

In [6]:
list_m = [100, 50, 400, 500]
list_m[1] = 200
list_m.append(600)
print(list_m)
list_m.insert(2,300)
list_m.remove(600)
list_m.pop(0)

[100, 200, 400, 500, 600]


100

In [7]:
nums = [10,20,30,40,50]
total_sum = sum(nums)
average = total_sum / len(nums)
print(total_sum)
print(average)

150
30.0


In [8]:
nums = [45,12,89,2,67]
print(max(nums))
print(min(nums))

89
2


In [9]:
factors = [2,3,5,7]
product = 1
for x in factors:
    product *= x
print(f"Product:{product}")

Product:210


In [10]:
numbers = [10,21,4,45,66,93,11]
even_count = 0
odd_count = 0
for num in numbers:
    if num %2 == 0:
        even_count += 1
    else:
        odd_count += 1
print(f"Even numbers: {even_count}")
print(f"Odd numbers: {odd_count}")

Even numbers: 3
Odd numbers: 4
